In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [13]:
df = pd.read_csv('2번문제/Multiple Regression Datasets.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76000 entries, 0 to 75999
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                76000 non-null  object 
 1   Store ID            76000 non-null  object 
 2   Product ID          76000 non-null  object 
 3   Category            76000 non-null  object 
 4   Region              76000 non-null  object 
 5   Inventory Level     76000 non-null  int64  
 6   Units Sold          76000 non-null  int64  
 7   Units Ordered       76000 non-null  int64  
 8   Price               76000 non-null  float64
 9   Discount            76000 non-null  int64  
 10  Weather Condition   76000 non-null  object 
 11  Promotion           76000 non-null  int64  
 12  Competitor Pricing  76000 non-null  float64
 13  Seasonality         76000 non-null  object 
 14  Epidemic            76000 non-null  int64  
 15  Demand              76000 non-null  int64  
dtypes: f

In [14]:
# 문자열 형태인 날짜를 datetime 객체로 변환
df['Date'] = pd.to_datetime(df['Date'])

# 요일 컬럼 추가 (0=월요일, 6=일요일)
df['Weekday'] = df['Date'].dt.weekday

# 원래 날짜 컬럼은 제거 (원할 경우 유지 가능)
df = df.drop('Date', axis=1)

In [15]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print("문자형 변수 목록:", cat_cols)

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

문자형 변수 목록: ['Store ID', 'Product ID', 'Category', 'Region', 'Weather Condition', 'Seasonality']


In [16]:
print(df.head())
print(df.dtypes)

   Store ID  Product ID  Category  Region  Inventory Level  Units Sold  \
0         0           0         1       1              195         102   
1         0           1         0       1              117         117   
2         0           2         0       1              247         114   
3         0           3         1       1              139          45   
4         0           4         3       1              152          65   

   Units Ordered  Price  Discount  Weather Condition  Promotion  \
0            252  72.72         5                  2          0   
1            249  80.16        15                  2          1   
2            612  62.94        10                  2          1   
3            102  87.63        10                  2          0   
4            271  54.41         0                  2          0   

   Competitor Pricing  Seasonality  Epidemic  Demand  Weekday  
0               85.73            3         0     115        5  
1               92.02   

In [17]:
# 예측 대상은 'Demand', 나머지는 입력값
X = df.drop('Demand', axis=1)
y = df['Demand']

In [18]:
# traing,test dataset 분할 80%:20% random_state 100 설정
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=100
)

In [19]:
print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

X_train shape: (60800, 15)
X_test shape : (15200, 15)
y_train shape: (60800,)
y_test shape : (15200,)


In [20]:
# 1. Linear Regression (파라미터 없음)
lr = LinearRegression()

# 2. Ridge Regression
ridge = Ridge()
ridge_params = {'alpha': [0.1, 1.0, 10.0]}

# 3. Random Forest
rf = RandomForestRegressor(random_state=100)
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20]
}

In [21]:
# Ridge Regression
grid_ridge = GridSearchCV(ridge, ridge_params, cv=3, scoring='neg_mean_absolute_error')
grid_ridge.fit(X_train, y_train)

GridSearchCV(cv=3, estimator=Ridge(), param_grid={'alpha': [0.1, 1.0, 10.0]},
             scoring='neg_mean_absolute_error')

In [22]:
# Random Forest Regression
grid_rf = GridSearchCV(rf, rf_params, cv=3, scoring='neg_mean_absolute_error')
grid_rf.fit(X_train, y_train)

KeyboardInterrupt: 

In [ ]:
# Linear Regression (그대로 학습)
lr.fit(X_train, y_train)

In [ ]:
# MAPE 함수 직접 정의
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # 0으로 나누는 것 방지
    return np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1e-8))) * 100

In [ ]:
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1e-8))) * 100

def evaluate_model(y_test, y_pred):
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = mse ** 0.5
    mape = mean_absolute_percentage_error(y_test, y_pred)
    return mae, mse, rmse, mape

# 각 모델 예측
y_pred_lr = lr.predict(X_test)
y_pred_ridge = grid_ridge.best_estimator_.predict(X_test)
y_pred_rf = grid_rf.best_estimator_.predict(X_test)

# 평가
results = {
    "LinearRegression": evaluate_model(y_test, y_pred_lr),
    "RidgeRegression": evaluate_model(y_test, y_pred_ridge),
    "RandomForest": evaluate_model(y_test, y_pred_rf)
}

# 보기 좋게 출력
metrics_df = pd.DataFrame(results, index=['MAE', 'MSE', 'RMSE', 'MAPE']).T
print(metrics_df)

In [ ]:
# 지표별 비교 시각화
metrics_df.plot(kind='bar', figsize=(10, 6))
plt.title("Regression Model Performance Comparison")
plt.ylabel("Error")
plt.xticks(rotation=0)
plt.grid(axis='y')
plt.legend(title='Metric')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred_rf, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')  # 대각선
plt.xlabel('Actual Demand')
plt.ylabel('Predicted Demand')
plt.title('Random Forest: Actual vs Predicted Demand')
plt.grid(True)
plt.tight_layout()
plt.show()

In [23]:
!jupyter nbconvert --to script 백인호2번문제.ipynb

[NbConvertApp] Converting notebook 백인호2번문제.ipynb to script
[NbConvertApp] Writing 3847 bytes to 백인호2번문제.py
